# 02c — HyperIndex: Word Addressing (C)

**Requires the C kernel.** Install with `./install_c_kernel.sh`.  
Binary cells call `../../PtolC/ptolemy` via `system()` — build first with `cd PtolC && make`.

This notebook covers:
- The bijective base-95 Horner hash → golden-ratio scatter
- φ as the maximum-entropy hash multiplier
- Energy range [D_STAR, OMEGA_ZS] and the 4 values of d*
- The cross-language property: **β convergence via A-coupling** after corpus learning

Parallel Python notebook: `../02_hyperindex_septuagint.ipynb`

## 1. The Hash Function

```
surface  →  n = Σᵢ char[i] × 95^i          bijective base-95 Horner
seed     =  fmod(n × φ, 1.0)               golden-ratio scatter → [0, 1)
idx      =  floor(seed × N)                 zero index in [0, N)
E        =  D_STAR + seed × (Ω − D_STAR)   spectral energy
γ        =  zeros[idx]                      Riemann zero address
σ        =  ½                               forced by Noether balance
```

`D_STAR = 0.24600` is the ℂ-projection of d* — a 4-component object. All 4 values are needed to perform ln(10) in radial complex spherical polar space (see notebook 03c).

In [ ]:
#include <stdio.h>
#include <math.h>

#define N          25000
#define D_STAR     0.24600
#define OMEGA_ZS   0.56714
#define PHI        1.6180339887498948

static void word_coords(const char *s, int *idx, double *E) {
    unsigned long long n = 0, base = 1;
    for (const char *p = s; *p; p++) {
        n += (unsigned char)(*p) * base;
        base *= 95;
    }
    double seed = fmod((double)n * PHI, 1.0);
    *idx = (int)(seed * N);
    *E   = D_STAR + seed * (OMEGA_ZS - D_STAR);
}

int main(void) {
    const char *words[] = {
        "water", "fire", "earth", "mind", "reason",
        "prime", "zero", "language", "mathematics",
        "eau", "aqua", "wasser", "agua",
        NULL
    };
    printf("word_coords() — bijective Horner + golden-ratio scatter (N=%d)\n\n", N);
    printf("  %-14s  %6s  %8s\n", "word", "idx", "E");
    printf("  %-14s  %6s  %8s\n", "--------------", "------", "--------");
    for (int i = 0; words[i]; i++) {
        int idx; double E;
        word_coords(words[i], &idx, &E);
        printf("  %-14s  %6d  %8.4f\n", words[i], idx, E);
    }
    printf("\n  Energy range: [D_STAR=%.5f, OMEGA_ZS=%.5f]\n", D_STAR, OMEGA_ZS);
    return 0;
}

## 2. Cross-Language Property — What it Is

The hash function maps different surface forms to **different** zero indices.  
`water` (z#9361, E=0.3663), `eau` (z#14459, E=0.4317), `aqua` (z#9404, E=0.3668), `wasser` (z#9912, E=0.3733) are at different addresses.

The cross-language property lives in the **β field**, not the hash:

1. `N = 25000` zeros + the English WordNet corpus builds a dense A-coupling matrix (766K edges)
2. `learn()` couples co-occurring zeros: `A[i,j] += E_i × E_j / (|γ_i − γ_j| + GAP)`
3. `speak()` propagates the Noether current through A — β deepened at `water` (z#9361) flows to connected zeros including those used by other languages

After learning the English corpus, querying in French, Latin, or German activates zeros that are A-coupled to the English concept.

In [ ]:
#include <stdio.h>

/* Adjust path if ptolemy is not at ../../PtolC/ptolemy */
#define PTOLEMY "../../PtolC/ptolemy"

int main(void) {
    /* Show actual hash output — different zeros for different languages */
    printf("Hash addresses for 'water' in multiple languages:\n");
    printf("(N=25000, monad_wordnet.bin — English WordNet corpus)\n\n");
    system(PTOLEMY " -q water  2>/dev/null");
    system(PTOLEMY " -q eau    2>/dev/null");
    system(PTOLEMY " -q aqua   2>/dev/null");
    system(PTOLEMY " -q wasser 2>/dev/null");
    printf("\nDifferent zero indices. The cross-language property is in beta\n");
    printf("convergence via A-coupling — not in hash identity.\n");
    printf("\nbeta values shown above reflect depth from English corpus.\n");
    printf("'water' is deep (English WordNet). 'eau','wasser' are shallow\n");
    printf("(not in WordNet but still addressed at their Riemann zeros).\n");
    printf("A parallel corpus (English+French) deepens both at their zeros\n");
    printf("and builds A-edges connecting them.\n");
    return 0;
}

## 3. A-Coupling: How the Cross-Language Connection Forms

When two words co-occur in a sentence, their zeros get coupled:

```c
A[i][j] += E_i * E_j / (fabs(gamma_i - gamma_j) + GAP)
```

`GAP = 0.000707` is the Yang-Mills mass gap — prevents divergence when zeros are nearby.

After a parallel corpus, `water` (z#9361) and `eau` (z#14459) are A-connected even though they hash to different zeros. The Noether current then propagates across that connection.

In [ ]:
#include <stdio.h>
#include <math.h>

#define GAP  0.000707

int main(void) {
    /* Actual gamma values from ptolemy -q output */
    struct { const char *word; int idx; double gamma; double E; } words[] = {
        {"water",  9361,  9331.32, 0.3663},
        {"eau",   14459, 13600.27, 0.4317},
        {"aqua",   9404,  9368.30, 0.3668},
        {"wasser", 9912,  9803.75, 0.3733},
        {"agua",      0,     0.00, 0.0000},   /* unknown — example only */
    };
    int n = 4;   /* skip agua — no gamma known */

    printf("A-coupling weights between 'water' and other language forms:\n");
    printf("  A[i,j] = E_i * E_j / (|gamma_i - gamma_j| + GAP)\n\n");
    printf("  %-10s  %10s  %10s  %12s\n",
           "pair", "|delta_g|", "E_i*E_j", "A weight");
    printf("  %-10s  %10s  %10s  %12s\n",
           "----------", "----------", "----------", "------------");

    for (int j = 1; j < n; j++) {
        double dg    = fabs(words[0].gamma - words[j].gamma);
        double EiEj  = words[0].E * words[j].E;
        double A     = EiEj / (dg + GAP);
        printf("  water/%-4s  %10.2f  %10.4f  %12.6f\n",
               words[j].word, dg, EiEj, A);
    }
    printf("\nThe coupling weight decays with distance in gamma space.\n");
    printf("In a parallel corpus, these A-edges are built by co-occurrence.\n");
    printf("The Noether current then flows across language boundaries.\n");
    return 0;
}

## 4. Why φ = 1.618034…

The golden ratio is the maximum-entropy hash multiplier. Its continued fraction `[1; 1, 1, 1, ...]` converges slowest — no rational approximation clusters inputs. Two words differing by one character land in different parts of the zero space.

In [ ]:
#include <stdio.h>
#include <math.h>

#define BUCKETS  256
#define WORDS    500

static double entropy_spread(double mult) {
    int occupied = 0;
    int counts[BUCKETS];
    for (int i = 0; i < BUCKETS; i++) counts[i] = 0;
    for (int i = 0; i < WORDS; i++) {
        double seed = fmod((double)(i * 97 + 31) * mult, 1.0);
        if (seed < 0) seed += 1.0;
        int b = (int)(seed * BUCKETS);
        if (b < 0) b = 0;
        if (b >= BUCKETS) b = BUCKETS - 1;
        counts[b]++;
    }
    for (int i = 0; i < BUCKETS; i++)
        if (counts[i] > 0) occupied++;
    return (double)occupied;
}

int main(void) {
    printf("Hash multiplier comparison (%d buckets, %d words):\n\n",
           BUCKETS, WORDS);
    printf("  %-28s  %s\n", "multiplier", "buckets occupied");
    printf("  %-28s  %s\n", "----------------------------", "----------------");

    struct { const char *label; double val; } mults[] = {
        {"1.0 (integer)",           1.0},
        {"1.5 (3/2 rational)",      1.5},
        {"1.6 (8/5 Fibonacci)",     1.6},
        {"1.618034 (truncated phi)",1.618034},
        {"phi (full precision)",     1.6180339887498948},
        {"1.619 (near phi)",        1.619},
        {"sqrt(2)=1.41421",         1.41421356237}
    };
    for (int i = 0; i < 7; i++)
        printf("  %-28s  %.0f\n",
               mults[i].label, entropy_spread(mults[i].val));

    printf("\nFull-precision phi occupies most buckets — maximum spread.\n");
    printf("Rational multipliers cluster — some buckets get many hits.\n");
    return 0;
}

## 5. Energy Distribution Over N=25000 Zeros

With N=25000, the energy range [0.24600, 0.56714] is divided into 25000 equal-width buckets.  
Bucket width = (Ω − D*) / N ≈ 1.3×10⁻⁵.

In [ ]:
#include <stdio.h>
#include <math.h>

#define N          25000
#define D_STAR     0.24600
#define OMEGA_ZS   0.56714329
#define PHI        1.6180339887498948

static void word_coords(const char *s, int *idx, double *E) {
    unsigned long long n = 0, base = 1;
    for (const char *p = s; *p; p++) {
        n += (unsigned char)(*p) * base;
        base *= 95;
    }
    double seed = fmod((double)n * PHI, 1.0);
    *idx = (int)(seed * N);
    *E   = D_STAR + seed * (OMEGA_ZS - D_STAR);
}

int main(void) {
    double bucket_width = (OMEGA_ZS - D_STAR) / N;
    printf("Energy range: [%.5f, %.5f]\n", D_STAR, OMEGA_ZS);
    printf("Bucket width at N=%d: %.2e\n\n", N, bucket_width);

    const char *words[] = {
        "the", "a", "of",
        "water", "eau", "aqua", "wasser",
        "mind", "reason", "consciousness",
        "prime", "zero", "mathematics",
        NULL
    };
    printf("  %-16s  %6s  %8s\n", "word", "idx", "E");
    printf("  %-16s  %6s  %8s\n", "----------------", "------", "--------");
    for (int i = 0; words[i]; i++) {
        int idx; double E;
        word_coords(words[i], &idx, &E);
        printf("  %-16s  %6d  %8.4f\n", words[i], idx, E);
    }
    printf("\nNote: water/aqua/wasser have close E values (~0.366-0.373)\n");
    printf("but land at different zero indices. A-coupling connects them\n");
    printf("after the English corpus is learned.\n");
    return 0;
}

## Summary

- Every word hashes deterministically to one Riemann zero. Pure function, no state, no lookup.
- φ maximises entropy: no rational clustering. Two words differing by one char land in different buckets.
- Cross-language property is in the **β field after corpus learning**, not the hash.
  - `water` (z#9361) and `eau` (z#14459) hash to different zeros.
  - After a parallel corpus, A-coupling connects them. Noether current crosses the connection.
- N=25000 + English WordNet is the production pair. This is where cross-language β convergence manifests.
- Ptolemy can increase N; rebuilding the checkpoint extends the address space.

**Next:** `03c_self_adjoint_hamiltonian.ipynb` — H_hat_RB and the four values of d*.